# ADA Semester Project

This represents the project for the CS-401 course at EPFL, done by:

| SCIPER                                              | Name                      |
|:----------------------------------------------------|:--------------------------|
| [356420](https://people.epfl.ch/badr.almahouri)     | Badr Al Mahouri           |
| [341237](https://people.epfl.ch/louis.grange)       | Louis Grange              |
| [340497](https://people.epfl.ch/daniel.alvesataide) | Daniel Ataíde             |
| [329310](https://people.epfl.ch/arnaud.tadic)       | Arnaud Jacques Yves Tadic |
| [308396](https://people.epfl.ch/yuri.cho)           | Yuri Cho                  |

## Install dependencies

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import math
#load the statistical libraries
from statsmodels.stats import diagnostic
from scipy import stats

## Import dataset

We can first import the data from the downloaded dataset, and format it into a correct Pandas Dataframe. If you didn't download it, just do it [here](https://snap.stanford.edu/data/soc-RedditHyperlinks.html).

In [ ]:
import src.utils as utils
from src.consts import features

In [ ]:
body_data, title_data, subreddits_data, users_data = utils.load_data()

In [ ]:
print("Number of data entries")
print(f"title: {len(title_data)}, body: {len(body_data)}, subreddits: {len(subreddits_data)}, users: {len(users_data)}")

In [ ]:
subreddits_data.head(5)

In [ ]:
title_data['SOURCE_SUBREDDIT'].value_counts()[:500]

In [ ]:
first_col_subreddits = subreddits_data.iloc[:, 0].tolist()
print(len(title_data.query(f"SOURCE_SUBREDDIT in {first_col_subreddits}")))

In [ ]:
subreddits_data.iloc[:, 1:]

In [ ]:
import sklearn
from sklearn.manifold import TSNE

# Initialize t-SNE
tsne = TSNE(n_components=2, random_state=42)

# Fit and transform
tsne_result = tsne.fit_transform(subreddits_data.iloc[:, 1:])  # Exclude the first column if it's an identifier

# Make a new DataFrame for the 2D embedding
tsne_df = pd.DataFrame(tsne_result, columns=['TSNE1', 'TSNE2'])

plt.figure(figsize=(8,6))
plt.scatter(tsne_df['TSNE1'], tsne_df['TSNE2'], s=1, alpha=0.7)
plt.title('t-SNE Map')
plt.xlabel('TSNE1')
plt.ylabel('TSNE2')
plt.show()

In [ ]:
tsne_df['SOURCE_SUBREDDIT'] = subreddits_data.iloc[:, 0].values
print(tsne_df['SOURCE_SUBREDDIT'])
tsne_df

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(tsne_df['TSNE1'], tsne_df['TSNE2'], s=1, alpha=0.7)
plt.title('t-SNE Map')
plt.xlabel('TSNE1')
plt.ylabel('TSNE2')

# Highlight specific subreddits
# top_subreddits = title_data['SOURCE_SUBREDDIT'].value_counts().head(100).index.tolist()
# highlight_df = tsne_df[tsne_df['SOURCE_SUBREDDIT'].isin(top_subreddits)]

sports_subreddits = ["soccer", "hockey", "nba"]
highlight_df = tsne_df[tsne_df['SOURCE_SUBREDDIT'].isin(sports_subreddits)]
print(highlight_df)
# Plot all points
plt.scatter(tsne_df['TSNE1'], tsne_df['TSNE2'], s=1, alpha=1, color='lightgray', edgecolor=None)
plt.title('t-SNE Map')
plt.xlabel('TSNE1')
plt.ylabel('TSNE2')

# Highlight specific subreddits
plt.scatter(
    highlight_df['TSNE1'],
    highlight_df['TSNE2'],
    s=2,
    alpha=1,
    color='red',
)

for i, row in highlight_df.iterrows():
    plt.text(
        row['TSNE1'],
        row['TSNE2'],
        row['SOURCE_SUBREDDIT'],
        fontsize=12,
        alpha=0.75
    )

plt.show()

In [ ]:
len(features), features[:10]

and then check if the data has been correctly imported and formatted

In [ ]:
title_data_features = title_data["PROPERTIES"].apply(lambda s: [float(x) for x in s.split(",")])

# Create a new DataFrame with features as columns
features_df = pd.DataFrame(title_data_features.tolist(), columns=features, index=title_data.index)

# Merge back into original DataFrame (drop old PROPERTIES if not needed)
title_data_expanded = pd.concat([title_data.drop(columns=["PROPERTIES"]), features_df], axis=1)

In [ ]:
title_data_expanded.head(5)

In [ ]:
body_data_features = body_data["PROPERTIES"].apply(lambda s: [float(x) for x in s.split(",")])

# Create a new DataFrame with features as columns
features_df = pd.DataFrame(body_data_features.tolist(), columns=features, index=body_data.index)

# Merge back into original DataFrame (drop old PROPERTIES if not needed)
body_data_expanded = pd.concat([body_data.drop(columns=["PROPERTIES"]), features_df], axis=1)

In [ ]:
body_data_expanded.head(5)

In [ ]:
# Check mean, max, sum of all features for a given dataframe
df = title_data_expanded
raw_rows =[]
for f in features[18:]:
    raw_rows.append((
        f,
        round(df[f].mean(), 5),
        round(df[f].max(), 2),
        round(df[f].sum(), 2)
    ))
features_mean_max_sum = pd.DataFrame(raw_rows, columns=["Feature", "Mean", "Max", "Sum"]).set_index("Feature")
features_mean_max_sum

## Examples of initial analysis

In [ ]:
keyword = "LIWC_Health"
df = title_data_expanded
# df = body_data_expanded

period = "M"  # "W" for weekly, "M" for monthly aggregation

df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])

print(df[keyword].mean())

sub_df = df[df[keyword] > 0]
print(f"Number of mentions of '{keyword}': {len(sub_df)}")

# Group by week or month depending on 'period'
sub_df = sub_df.assign(PERIOD=sub_df["TIMESTAMP"].dt.to_period(period))

# Sum of keyword values per period
sum_by_period = sub_df.groupby("PERIOD")[keyword].sum()

# Count of mentions per period
count_by_period = sub_df.groupby("PERIOD").size()

# Plot both on subplots
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Plot 1: Sum of keyword values
sum_by_period.plot(kind="bar", ax=axes[0], color="skyblue")
axes[0].set_title(f"Sum of '{keyword}' by {'Week' if period == 'W' else 'Month'}")
axes[0].set_ylabel("Sum of keyword values")

# Plot 2: Count of mentions
count_by_period.plot(kind="bar", ax=axes[1], color="lightcoral")
axes[1].set_title(f"Mentions of '{keyword}' by {'Week' if period == 'W' else 'Month'}")
axes[1].set_xlabel("Period")
axes[1].set_ylabel("Number of mentions")

plt.tight_layout()
plt.show()


In [ ]:
df = title_data_expanded
col = "TARGET_SUBREDDIT"

# Get counts
counts = df[col].value_counts()
thres = 1000

# Filter subreddits with more than threshold occurrences
counts_filtered = counts[counts > thres]

# Calculate total for percentage comparison
total = counts_filtered.sum()

cutoff = 1  # percentage cutoff
# Custom function to hide small slices in pie chart
def autopct_hide(pct):
    return f"{pct:.1f}%" if pct >= cutoff else ""

# For labels: show only if the slice is above cutoff percentage
labels = [
    label if (value / total * 100) >= cutoff else ""
    for label, value in zip(counts_filtered.index, counts_filtered)
]

# Plot pie chart
plt.figure(figsize=(8, 8))
plt.pie(
    counts_filtered,
    labels=labels,
    autopct=autopct_hide,
    startangle=90,
    counterclock=False
)
print(counts_filtered.index[:10])
plt.title(f"{col} (more than {thres} occurrences)", fontsize=14)
plt.show()

In [ ]:
subreddit_list = ["apple"]
# subreddit_list = ["samsung", "apple"]

# Combine all keywords into one regex pattern
pattern = "|".join(subreddit_list)

# Add a new boolean column to each dataset
title_data_expanded["is_relevant"] = (
    title_data_expanded["SOURCE_SUBREDDIT"].str.contains(pattern, case=False, na=False) |
    title_data_expanded["TARGET_SUBREDDIT"].str.contains(pattern, case=False, na=False)
)

body_data_expanded["is_relevant"] = (
    body_data_expanded["SOURCE_SUBREDDIT"].str.contains(pattern, case=False, na=False) |
    body_data_expanded["TARGET_SUBREDDIT"].str.contains(pattern, case=False, na=False)
)

# Optional: check how many matched
print("title_data_expanded relevant count:", title_data_expanded["is_relevant"].sum())
print("body_data_expanded relevant count:", body_data_expanded["is_relevant"].sum())


In [ ]:
title_data_expanded[title_data_expanded["is_relevant"]].head(5)

In [ ]:
# Histogram of relevant mentions over time for a given dataframe
pattern = "apple"
tf = title_data_expanded

# Add a new boolean column to each dataset
tf["is_relevant"] = (
    tf["SOURCE_SUBREDDIT"].str.contains(pattern, case=False, na=False) |
    tf["TARGET_SUBREDDIT"].str.contains(pattern, case=False, na=False)
)

tf["TIMESTAMP"] = pd.to_datetime(tf["TIMESTAMP"])

# df =  title_data_expanded[title_data_expanded["is_relevant"]]
df = tf[
    (tf["is_relevant"])
    & (tf["TIMESTAMP"].dt.year >= 2016)
    & (tf["TIMESTAMP"].dt.year <= 2017)
]
period = "W"

if period == "M":
    df = df.assign(MONTH=df["TIMESTAMP"].dt.to_period("M"))
    counts_by_month = df.groupby("MONTH").size()
    counts_by_month.plot(kind="bar", figsize=(12,5))
    plt.title(f"Occurrences of subreddit containing '{pattern}' by Month in Dataframe (2016.01–2017.04)")
    plt.xlabel("Month")

elif period == "W":
    df = df.assign(WEEK=df["TIMESTAMP"].dt.to_period("W"))
    counts_by_week = df.groupby("WEEK").size()
    counts_by_week.plot(kind="bar", figsize=(16,5))
    plt.title(f"Occurrences of subreddit containing '{pattern}' by Week in Dataframe (2016.01–2017.04)")
    plt.xlabel("Week")
    # print(counts_by_week)
plt.ylabel("Count")
plt.show()

In [ ]:
pattern = "apple"
tf = title_data_expanded

# Add a new boolean column to each dataset
tf["is_relevant"] = (
    tf["SOURCE_SUBREDDIT"].str.contains(pattern, case=False, na=False) |
    tf["TARGET_SUBREDDIT"].str.contains(pattern, case=False, na=False)
)

tf["TIMESTAMP"] = pd.to_datetime(tf["TIMESTAMP"])
df = tf[
    (tf["is_relevant"])
    & (tf["TIMESTAMP"].dt.year >= 2016)
    & (tf["TIMESTAMP"].dt.year <= 2017)
]
# Assign weekly period
df = df.assign(WEEK=df["TIMESTAMP"].dt.to_period("W"))

# Group by week: average sentiment and count
sentiment_cols = [
    "Positive sentiment (VADER)",
    "Negative sentiment (VADER)",
    "Compound sentiment (VADER)"
]

weekly_stats = (
    df.groupby("WEEK")[sentiment_cols]
    .sum()
    .assign(Count=df.groupby("WEEK").size())
)

# Convert period index to datetime
weekly_stats.index = weekly_stats.index.to_timestamp()

# --- Plot setup ---
fig, ax1 = plt.subplots(figsize=(16,6))

# Bar plot for post count
ax1.bar(weekly_stats.index, weekly_stats["Count"], color="gray", label="Post Count")
ax1.set_ylabel("Post Count", color="gray")
ax1.set_xlabel("Week")
ax1.tick_params(axis="y", labelcolor="gray")

# Line plot for each sentiment on the second axis
ax2 = ax1.twinx()
ax2.plot(weekly_stats.index, weekly_stats["Positive sentiment (VADER)"], color="blue", label="Positive Sentiment", linewidth=2)
ax2.plot(weekly_stats.index, weekly_stats["Negative sentiment (VADER)"], color="red", label="Negative Sentiment", linewidth=2)
ax2.plot(weekly_stats.index, weekly_stats["Compound sentiment (VADER)"], color="gold", label="Compound Sentiment", linewidth=2)

ax2.set_ylabel("Sum Sentiment Score")
ax2.tick_params(axis="y")

# Title and legend
plt.title(f"Mentions of '{pattern}' and Sentiment Trends (2016.01–2017.04)")
fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
fig.tight_layout()
plt.show()


In [ ]:
weekly_stats

## etc

In [ ]:
# Check number of entries for specific subreddits
subreddit_list = ["politics", "worldnews", "news", "bitcoin", "sandersforpresident", "economics", "Conservative", "Republican" "trump", "usa"]
for subreddit in subreddit_list:
    print(f"{subreddit=}")
    print(f"number of entries: {len(title_data[title_data['SOURCE_SUBREDDIT'] == subreddit]),
                                len(title_data[title_data['TARGET_SUBREDDIT'] == subreddit]),
                                len(body_data[body_data['SOURCE_SUBREDDIT'] == subreddit]),
                                len(body_data[body_data['TARGET_SUBREDDIT'] == subreddit])}\n"
                                )

In [ ]:
# subreddit_list = ["politics", "news", "bitcoin", "sandersforpresident", "economics", "Conservative", "Republican", "trump", "usa"]
subreddit_list = ["apple", "samsung"]
for check in subreddit_list:
    print(f"\n=== {check.upper()} ===")

    # Source subreddit matches
    print("SOURCE_SUBREDDIT (titles):")
    print(title_data_expanded[title_data_expanded["SOURCE_SUBREDDIT"].str.contains(check, case=False, na=False)]["SOURCE_SUBREDDIT"].value_counts())

    print("TARGET_SUBREDDIT (titles):")
    print(title_data_expanded[title_data_expanded["TARGET_SUBREDDIT"].str.contains(check, case=False, na=False)]["TARGET_SUBREDDIT"].value_counts())

    print("SOURCE_SUBREDDIT (bodies):")
    print(body_data_expanded[body_data_expanded["SOURCE_SUBREDDIT"].str.contains(check, case=False, na=False)]["SOURCE_SUBREDDIT"].value_counts())

    print("TARGET_SUBREDDIT (bodies):")
    print(body_data_expanded[body_data_expanded["TARGET_SUBREDDIT"].str.contains(check, case=False, na=False)]["TARGET_SUBREDDIT"].value_counts())